# 04 · Practical B — The Numerical Integration Pricer

**Read first:** Chapter 5 (*The Black-Scholes Framework*), then Practical B.

---

## What you'll be able to do after this

- Build the terminal spot distribution from spot, two interest rates, time and volatility — and explain what each input does to its shape.
- Price *any* payoff that depends only on spot at maturity, by integrating that payoff against the distribution.
- Read the single chart this practical exists to produce: the payoff laid over the density, which is what an option price actually *is*.

## The intuition, before the maths

Forget formulas for a moment.

You want to know what a bet is worth. The bet pays out depending on where the exchange rate ends up in a year's time. Two things decide its value:

1. **How likely is each outcome?**
2. **What does the bet pay in that outcome?**

Multiply those together for every outcome, add them up, and you have the expected payout. Then adjust for the fact that you get paid in a year rather than today. That's the whole practical. Everything else is bookkeeping.

### A worked example with round numbers

Suppose spot is **100** and there are exactly three things that can happen in a year:

| Outcome | Probability | Spot ends at | A 100-strike call pays |
|---|---|---|---|
| Down | 25% | 90 | 0 |
| Flat | 50% | 100 | 0 |
| Up | 25% | 110 | 10 |

Expected payout = `0.25 × 0 + 0.50 × 0 + 0.25 × 10` = **2.5**.

That's the option's value at maturity. If interest rates were positive you'd discount it back to today, and that's the price.

Real spot doesn't have three outcomes — it has a continuum. So we chop the continuum into 100 thin buckets, work out the probability of landing in each, and do exactly the same sum. That is *numerical integration*, and the fancy name hides how simple the idea is.

## The maths, derived not asserted

### Why log-normal, and where the odd `−σ²/2` comes from

Chapter 5 models spot with

$$\frac{dS_t}{S_t} = (r_{CCY2} - r_{CCY1})\,dt + \sigma\,dW_t$$

Note the left side: it's a **relative** change, `dS/S`, not an absolute one. That choice has a consequence worth stating plainly — as spot gets closer to zero, its moves get smaller in absolute terms, so **spot can never reach zero**. That matches FX (unlike, say, an equity, which genuinely can go to zero).

Two pieces on the right:

- **Drift**, `(r₂ − r₁)dt` — deterministic. Set `σ = 0` and this alone gives `F_T = S·e^((r₂−r₁)T)`, the forward. Worth pausing on: *zero volatility does not mean spot is static.* It means spot follows the forward path exactly.
- **Uncertainty**, `σ dW_t` — a Wiener process scaled by volatility.

Solving it (Itō's lemma does the work) gives the log return to expiry:

$$\ln\!\left(\frac{S_T}{S_0}\right) = \underbrace{\left(r_{CCY2} - r_{CCY1} - \frac{\sigma^2}{2}\right)T}_{\mu,\ \text{expected log return}} + \underbrace{\sigma W_T}_{\text{normal, sd } \sigma\sqrt{T}}$$

The `−σ²/2` is the **Itō correction**. It's not a fudge factor. It's there because the *average of the exponential* is not the *exponential of the average* — pushing a symmetric distribution through `exp()` skews it upward, and this term subtracts exactly that skew so the forward comes out right. You'll verify it numerically in Experiment 4.

Every symbol, defined once:

| Symbol | Meaning | Units |
|---|---|---|
| $S$ | spot now | CCY2 per CCY1 |
| $S_T$ | spot at expiry | CCY2 per CCY1 |
| $K$ | strike | CCY2 per CCY1 |
| $T$ | time to expiry | years |
| $r_{CCY1}, r_{CCY2}$ | continuously compounded rates | decimal |
| $\sigma$ | volatility of log returns | decimal (0.10 = 10%) |
| $\mu$ | expected log return over $T$ | dimensionless |
| $N(x)$ | cumulative standard normal | probability |

### Building the grid

Standard deviation of the log return is $\sigma\sqrt{T}$. Step from −5 to +5 standard deviations in 0.1 increments — 101 points. Beyond ±5σ there's about 6 × 10⁻⁷ of probability left in both tails combined, which is nothing.

For each grid point $X$:

$$\text{return level} = \mu + X\,\sigma\sqrt{T}, \qquad \text{spot level} = S\,e^{\text{return level}}$$

### Bucket probabilities

The probability of landing between two grid points is the difference of two cumulative normals:

$$P(\text{bucket } i) = N(X_{i+1}) - N(X_i)$$

**Row alignment is the thing people get wrong.** The probability on row *i* belongs to the interval running from row *i* **to the next row**. The last row bounds no bucket, so it has no probability.

### The integration

$$\text{Value} = e^{-r_{CCY2}T}\sum_i P(\text{bucket } i)\cdot\frac{\text{payoff}_i + \text{payoff}_{i+1}}{2}$$

Using the **average** payoff across the bucket, rather than one endpoint, makes this the trapezoidal rule instead of a cruder rectangle sum. It's why 101 points gets within a quarter of a percent of the exact answer.

Then convert units: CCY2 pips ÷ spot = CCY1%.

## The code

The package implements; this notebook demonstrates. Nothing is redefined inline.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from fxds.conventions import OptionType
from fxds.numerical import (
    terminal_distribution,
    integrate_payoff,
    price_vanilla,
    call_payoff,
    put_payoff,
    long_forward_payoff,
)
from fxds.blackscholes import forward, price
from fxds.plotting import (
    use_house_style, style_axis, mark_level,
    DENSITY_COLOUR, PAYOFF_COLOUR, PRIMARY, SECONDARY, TERTIARY, MUTED,
)

use_house_style()
pd.set_option("display.precision", 6)

### Task A, Step 1 — the terminal spot distribution

The book's own reference case: spot 100, zero rates, one year, 10% volatility.

In [2]:
dist = terminal_distribution(spot=100.0, T=1.0, r_ccy1=0.0, r_ccy2=0.0, sigma=0.10)

print(f"grid points:        {len(dist)}")
print(f"spans:              {dist['sd'].min():+.1f} to {dist['sd'].max():+.1f} standard deviations")
print(f"spot range:         {dist['spot_level'].min():.2f} to {dist['spot_level'].max():.2f}")
print(f"probabilities sum to: {dist['probability'].sum():.8f}")
dist.head(3)

grid points:        101
spans:              -5.0 to +5.0 standard deviations
spot range:         60.35 to 164.05
probabilities sum to: 0.99999943


,sd,return_level,spot_level,probability
0,-5.0,-0.505,60.350558,1.925317e-07
1,-4.9,-0.495,60.957091,3.141449e-07
2,-4.8,-0.485,61.569720,5.074793e-07


Look at the middle of the table — this is where the row-alignment point becomes concrete.

In [3]:
dist.iloc[48:53][["sd", "return_level", "spot_level", "probability"]]

,sd,return_level,spot_level,probability
48,-2.000000e-01,-0.025,97.530991,0.039432
49,-1.000000e-01,-0.015,98.511194,0.039828
50,-1.776357e-14,-0.005,99.501248,0.039828
51,1.000000e-01,0.005,100.501252,0.039432
52,2.000000e-01,0.015,101.511306,0.038652


The probability `0.039695` on the row where `sd = -0.1` is the chance of finishing between **that** row's spot level and the **next** row's. It is not the probability of being at that point — a continuous distribution assigns zero probability to any single point.

### Task A, Step 2 — plot the density

In [4]:
fig, ax = plt.subplots()
ax.fill_between(dist["spot_level"], dist["probability"], color=DENSITY_COLOUR, alpha=0.85)
ax.plot(dist["spot_level"], dist["probability"], color=PRIMARY, linewidth=1.5)
mark_level(ax, 100.0, "spot = forward")

style_axis(
    ax,
    "Terminal spot distribution — 1 year, 10% volatility, zero rates",
    "Spot at expiry (CCY2 per CCY1)",
    "Probability of finishing in this bucket",
    "The bell shape is the log return being normal. Note the right tail runs further than the left: that is log-normality.",
)
plt.show()

### Task A — the four behaviours the book asks you to check

Change one input at a time and watch the distribution move. Predict each before you run it.

In [5]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Left: volatility and time both widen the distribution.
for label, (T, sig), colour in [
    ("3 months, 10% vol", (0.25, 0.10), TERTIARY),
    ("1 year, 10% vol",   (1.00, 0.10), PRIMARY),
    ("1 year, 20% vol",   (1.00, 0.20), SECONDARY),
]:
    d = terminal_distribution(100.0, T, 0.0, 0.0, sig)
    axes[0].plot(d["spot_level"], d["probability"], label=label, color=colour)

axes[0].set_xlim(50, 175)
axes[0].legend()
style_axis(
    axes[0], "Width: volatility and time", "Spot at expiry (CCY2 per CCY1)",
    "Bucket probability",
    "Shorter or lower vol tightens; longer or higher vol widens. Both act through σ√T.",
)

# Right: the rate differential shifts it via the forward.
for label, (r1, r2), colour in [
    ("r1 = 10%, r2 = 0%  (forward lower)", (0.10, 0.00), TERTIARY),
    ("r1 = r2 = 0%       (forward = spot)", (0.00, 0.00), PRIMARY),
    ("r1 = 0%, r2 = 10%  (forward higher)", (0.00, 0.10), SECONDARY),
]:
    d = terminal_distribution(100.0, 1.0, r1, r2, 0.10)
    axes[1].plot(d["spot_level"], d["probability"], label=label, color=colour)

axes[1].set_xlim(60, 165)
axes[1].legend()
style_axis(
    axes[1], "Location: the rate differential", "Spot at expiry (CCY2 per CCY1)",
    "Bucket probability",
    "Higher CCY2 or lower CCY1 rates push the forward — and the whole distribution — higher.",
)
plt.tight_layout()
plt.show()

### Task B — attach a payoff and integrate

Four payoffs, all in CCY2 pips (CCY2 per one CCY1):

| Structure | Payoff at maturity |
|---|---|
| Long forward | $S_T - K$ |
| Short forward | $K - S_T$ |
| Vanilla call | $\max(S_T - K, 0)$ |
| Vanilla put | $\max(K - S_T, 0)$ |

In [6]:
result = price_vanilla(OptionType.CALL, spot=100.0, strike=100.0, T=1.0,
                       r_ccy1=0.0, r_ccy2=0.0, sigma=0.10)

print(f"value at maturity (CCY2 pips):  {result.undiscounted_ccy2_pips:.6f}")
print(f"present valued  (CCY2 pips):    {result.value_ccy2_pips:.6f}")
print(f"as CCY1%:                       {result.value_ccy1_pct * 100:.4f}%")
print()
print("The working, around the strike:")
result.table.iloc[48:53][["spot_level", "probability", "payoff", "average_payoff", "weighted_payoff"]]

value at maturity (CCY2 pips):  3.996905
present valued  (CCY2 pips):    3.996905
as CCY1%:                       3.9969%

The working, around the strike:


,spot_level,probability,payoff,average_payoff,weighted_payoff
48,97.530991,0.039432,0.000000,0.000000,0.000000
49,98.511194,0.039828,0.000000,0.000000,0.000000
50,99.501248,0.039828,0.000000,0.250626,0.009982
51,100.501252,0.039432,0.501252,1.006279,0.039679
52,101.511306,0.038652,1.511306,2.021409,0.078131


### The chart the whole practical exists for

Density and payoff on the same axes. The option's value is the **overlap** — probability weighted by payoff, summed.

In [7]:
fig, ax = plt.subplots(figsize=(10, 5.5))
tbl = result.table

ax.fill_between(tbl["spot_level"], tbl["probability"], color=DENSITY_COLOUR,
                alpha=0.85, label="Probability of finishing here")
ax.set_ylim(bottom=0)
mark_level(ax, 100.0, "strike")

ax2 = ax.twinx()
ax2.plot(tbl["spot_level"], tbl["payoff"], color=PAYOFF_COLOUR, linewidth=2.5,
         label="Call payoff at maturity")
ax2.set_ylabel("Payoff (CCY2 pips)", color=PAYOFF_COLOUR)
ax2.tick_params(axis="y", colors=PAYOFF_COLOUR)
ax2.set_ylim(bottom=0)
ax2.grid(False)

# Shade where both are non-zero: that region IS the option's value.
overlap = tbl["spot_level"] >= 100.0
ax2.fill_between(tbl["spot_level"][overlap], tbl["payoff"][overlap],
                 color=PAYOFF_COLOUR, alpha=0.12)

ax.set_xlim(60, 165)
lines = ax.get_legend_handles_labels()[0] + ax2.get_legend_handles_labels()[0]
labels = ax.get_legend_handles_labels()[1] + ax2.get_legend_handles_labels()[1]
ax.legend(lines, labels, loc="upper right")

style_axis(
    ax,
    "Where an option price comes from",
    "Spot at expiry (CCY2 per CCY1)",
    "Probability of finishing in this bucket",
    "The price is the shaded overlap: how likely each outcome is, times what it pays. Everything below the strike pays nothing, however likely it is.",
)
plt.show()

Sit with that chart. It explains, without a single formula, why:

- **Higher volatility raises both calls and puts.** A wider bell pushes more probability into the region where the payoff is non-zero.
- **A strike further from the forward is cheaper.** You're moving the payoff's elbow out to where the bell is thin.
- **The distribution's location matters as much as its width.** Shifting the bell right (higher CCY2 rates) drags probability into the call's payoff region.

### Task B — the book's two acceptance tests

In [8]:
# Test 1: a forward struck at the forward is worth (approximately) nothing.
S, T, r1, r2, sig = 1.30, 1.0, 0.02, 0.05, 0.12
F = forward(S, T, r1, r2)
fwd_value = integrate_payoff(long_forward_payoff(F), S, T, r1, r2, sig).value_ccy2_pips

print(f"Test 1  forward to 1y:        {F:.6f}")
print(f"        value struck at F:    {fwd_value:.3e}  → approximately zero ✓")
print()

# Test 2: S = K = 100, zero rates, T = 1.0 → very slightly under 4.00 CCY1%.
t2 = price_vanilla(OptionType.CALL, 100.0, 100.0, 1.0, 0.0, 0.0, 0.10)
print(f"Test 2  call value:           {t2.value_ccy1_pct * 100:.4f} CCY1%")
print(f"        under 4.00?           {t2.value_ccy1_pct * 100 < 4.0} ✓")

Test 1  forward to 1y:        1.339591
        value struck at F:    3.044e-05  → approximately zero ✓

Test 2  call value:           3.9969 CCY1%
        under 4.00?           True ✓


> **A note on σ.** The book states Test 2's inputs as `S = K = 100`, `r₁ = r₂ = 0%`, `T = 1.0` and the expected answer as "very slightly under 4.00 CCY1%" — but doesn't state the volatility in the text. It's 10%: the closed-form ATM value is approximately `0.3989·σ·√T`, which gives 3.99% at σ = 10%, and that matches the figure Practical C uses. Recorded in `notes/deviations.md`.

### The headline cross-check

Practical B integrates numerically and knows nothing about Garman–Kohlhagen. Practical C evaluates a closed-form expression and knows nothing about grids. They share only the inputs and the log-normal assumption.

If they agree, both are almost certainly right — and the closed form is revealed as what it actually is: **the analytic solution to the integral you just computed by brute force.**

In [9]:
args = dict(spot=100.0, strike=100.0, T=1.0, r_ccy1=0.0, r_ccy2=0.0, sigma=0.10)
analytic = price(OptionType.CALL, **args)

rows = []
for step in [0.5, 0.2, 0.1, 0.05, 0.02, 0.01]:
    numeric = price_vanilla(OptionType.CALL, **args, sd_step=step).value_ccy2_pips
    rows.append({
        "sd_step": step,
        "grid points": int(10 / step) + 1,
        "integration": numeric,
        "closed form": analytic,
        "relative error": abs(numeric - analytic) / analytic,
    })

comparison = pd.DataFrame(rows)
print(comparison.to_string(index=False, float_format=lambda v: f"{v:.8f}"))
print()
print("The book's own grid is sd_step = 0.1 → agreement to about 0.2%.")
print("Halving the step cuts the error ~4x: second-order convergence, as the trapezoidal rule predicts.")

   sd_step  grid points  integration  closed form  relative error
0.50000000           21   4.13607921   3.98776117      0.03719331
0.20000000           51   4.01937000   3.98776117      0.00792646
0.10000000          101   3.99690481   3.98776117      0.00229293
0.05000000          201   3.98878868   3.98776117      0.00025767
0.02000000          501   3.98810861   3.98776117      0.00008713
0.01000000         1001   3.98778378   3.98776117      0.00000567

The book's own grid is sd_step = 0.1 → agreement to about 0.2%.
Halving the step cuts the error ~4x: second-order convergence, as the trapezoidal rule predicts.


This is asserted in `tests/test_cross_validation.py`, the headline test of the repository. Run `pytest tests/test_cross_validation.py -v` to see it across a range of parameters.

In [10]:
fig, ax = plt.subplots()
ax.loglog(comparison["sd_step"], comparison["relative error"], "o-", color=PRIMARY,
          label="observed error")

# A reference line of exact second-order slope, for comparison.
ref = comparison["relative error"].iloc[0] * (comparison["sd_step"] / comparison["sd_step"].iloc[0]) ** 2
ax.loglog(comparison["sd_step"], ref, "--", color=MUTED, linewidth=1.2, label="second-order reference")
ax.legend()

style_axis(
    ax,
    "The gap is discretisation, not disagreement",
    "Grid step (standard deviations, log scale)",
    "Relative error vs closed form (log scale)",
    "Parallel to the reference line means error ∝ step². Refine the grid and the two methods converge on the same number.",
)
plt.show()

## Experiments

For each: **write down your prediction first**, then run the cell.

### Experiment 1 — Quadruple the time, or double the volatility

The random term is `σ·W_T`, with standard deviation `σ√T`. So `T → 4T` multiplies the width by `√4 = 2`, and `σ → 2σ` also multiplies it by 2.

**Predict:** will a 4-year 10% option and a 1-year 20% option have the same value? Exactly the same, or only nearly?

In [11]:
base = price_vanilla(OptionType.CALL, 100.0, 100.0, 1.0, 0.0, 0.0, 0.10).value_ccy2_pips
four_years = price_vanilla(OptionType.CALL, 100.0, 100.0, 4.0, 0.0, 0.0, 0.10).value_ccy2_pips
double_vol = price_vanilla(OptionType.CALL, 100.0, 100.0, 1.0, 0.0, 0.0, 0.20).value_ccy2_pips

print(f"1 year,  10% vol:   {base:.6f}")
print(f"4 years, 10% vol:   {four_years:.6f}")
print(f"1 year,  20% vol:   {double_vol:.6f}")
print(f"\ndifference between the last two: {abs(four_years - double_vol):.2e}")

1 year,  10% vol:   3.996905
4 years, 10% vol:   7.975733
1 year,  20% vol:   7.975733

difference between the last two: 0.00e+00


**Result:** identical to numerical precision. With zero rates, volatility and time enter the distribution *only* through the product `σ√T`. Four times the time genuinely is the same as double the volatility.

This stops being exactly true once rates are non-zero — the drift term scales with `T`, not `√T`, so the two stop being interchangeable. Try it.

### Experiment 2 — Raise both interest rates to the same level

**Predict:** the forward is unchanged (the differential `r₂ − r₁` is what drives it). So is the option price unchanged?

In [12]:
for r in [0.00, 0.05, 0.10]:
    F = forward(1.0, 1.0, r, r)
    v = price_vanilla(OptionType.CALL, 1.0, 1.0, 1.0, r, r, 0.10).value_ccy2_pips
    print(f"r1 = r2 = {r:.0%}   forward = {F:.6f}   call = {v:.6f}")

r1 = r2 = 0%   forward = 1.000000   call = 0.039969
r1 = r2 = 5%   forward = 1.000000   call = 0.038020
r1 = r2 = 10%   forward = 1.000000   call = 0.036165


**Result:** the forward is pinned at 1.0, but the price *falls*. The distribution is untouched — what changes is the discount factor `e^(−r₂T)` applied to the expected payoff. You're being paid the same amount at maturity, but that future money is worth less today. This is Practical C, Task A, Example 3.

### Experiment 3 — Where does the tail actually matter?

**Predict:** the grid spans ±5 standard deviations. Truncating to ±3 loses about 0.27% of the probability. Does the price move by roughly 0.27%?

In [13]:
for rng in [2.0, 3.0, 4.0, 5.0, 8.0]:
    r = price_vanilla(OptionType.CALL, 100.0, 100.0, 1.0, 0.0, 0.0, 0.10,
                      sd_range=rng, sd_step=0.01)
    lost = 1.0 - r.total_probability
    print(f"±{rng:.0f} sd:  probability captured {r.total_probability:.6f}"
          f"  (missing {lost:.2e})   call = {r.value_ccy2_pips:.6f}")

±2 sd:  probability captured 0.954500  (missing 4.55e-02)   call = 3.391154
±3 sd:  probability captured 0.997300  (missing 2.70e-03)   call = 3.936211
±4 sd:  probability captured 0.999937  (missing 6.33e-05)   call = 3.986161
±5 sd:  probability captured 0.999999  (missing 5.73e-07)   call = 3.987784
±8 sd:  probability captured 1.000000  (missing 1.33e-15)   call = 3.987803


**Result:** and this is worth getting right, because the obvious guess is wrong.

Truncating at ±3σ loses 0.27% of the probability — but moves the price by **1.3%**, roughly five times as much. Losing tail probability costs a call *more* than proportionally, because the payoff out there is large. At +3σ spot is around 135 and the payoff is 35, versus an option worth 4. Each unit of probability you drop in the upper tail is multiplied by a big number before it hits the sum.

So why is ±5 enough? Not because tail probability is unimportant, but because by ±5σ there is only 6 × 10⁻⁷ of it left. Even multiplied by a large payoff, that's nothing — and pushing out to ±8 changes the sixth decimal place. The book's choice is doing real work; ±3 would not be safe.

Try this with the *forward* payoff instead of a call and the tails matter more still: a forward's payoff is unbounded in **both** directions, so there's no region where the payoff is zero to protect you. That's why `tests/test_cross_validation.py` measures the forward's error in absolute terms rather than relative.

In [14]:
S, T, r1, r2, sig = 100.0, 2.0, 0.01, 0.04, 0.25
d = terminal_distribution(S, T, r1, r2, sig, sd_step=0.001)

# Expected terminal spot: probability-weighted average of the bucket midpoints.
mid = (d["spot_level"] + d["spot_level"].shift(-1)) / 2
expected_ST = (d["probability"] * mid).sum() / d["probability"].sum()

print(f"forward       F = S·e^((r2-r1)T) = {forward(S, T, r1, r2):.6f}")
print(f"E[S_T] from the distribution     = {expected_ST:.6f}")
print()

# Now the counterfactual: drift WITHOUT the Ito correction.
mu_no_ito = (r2 - r1) * T
sd = sig * np.sqrt(T)
sd_grid = np.arange(-5, 5.001, 0.001)
spots_no_ito = S * np.exp(mu_no_ito + sd_grid * sd)
from scipy.stats import norm
probs = np.diff(norm.cdf(sd_grid))
mids_no_ito = (spots_no_ito[:-1] + spots_no_ito[1:]) / 2
print(f"E[S_T] if we drop the -σ²/2 term = {(probs * mids_no_ito).sum() / probs.sum():.6f}")
print(f"  overshoots by a factor of      {np.exp(sig**2 / 2 * T):.6f}  = e^(σ²T/2)")

forward       F = S·e^((r2-r1)T) = 106.183655
E[S_T] from the distribution     = 106.183534

E[S_T] if we drop the -σ²/2 term = 113.031784
  overshoots by a factor of      1.064494  = e^(σ²T/2)


**Result:** with the correction, expected terminal spot lands on the forward. Without it, it overshoots by exactly `e^(σ²T/2)`.

That's the correction earning its keep. Pushing a symmetric normal distribution through `exp()` produces a distribution whose *mean* sits above the exponential of the mean — the upside tail stretches further than the downside compresses. The `−σ²/2` subtracts precisely that excess. It's a bookkeeping term that keeps the model arbitrage-free: if expected spot didn't equal the forward, you could trade the difference.

## Common misconceptions

**"The probability column is the probability of spot being at that level."**
No. A continuous distribution assigns zero probability to any single point. Each entry is the probability of landing in the bucket **between that row and the next**. The last row has no probability at all, which is the giveaway.

**"Higher volatility helps calls and hurts puts."**
It helps both. Volatility widens the distribution symmetrically in log space — more probability reaches the call's payoff region *and* the put's. Direction is set by the drift, not the volatility. (Practical C, Task A, Example 2.)

**"Zero volatility means spot doesn't move."**
It means spot follows the forward path *exactly*. With `r₂ > r₁` and zero volatility, spot rises deterministically. Chapter 5 flags this explicitly.

**"Pips and percent are two ways of writing the same number."**
They aren't. CCY2 pips is CCY2 per one CCY1; CCY1% is a fraction of the CCY1 notional. Converting between them means dividing by spot. Getting this wrong scales your price by ~1.3 in EUR/USD and by ~100 in USD/JPY — which is at least obvious. In a pair trading near 1.0000 the error is small enough to hide.

**"The `−σ²/2` is a correction for something being wrong."**
It's what makes the model *right*. Without it the expected terminal spot wouldn't equal the forward, and the model would price in an arbitrage. Experiment 4 measures it.

**"More grid points is always better."**
Only up to a point. Refining the step helps until you hit the tail-truncation floor; widening past ±5σ does nothing at all. The error curve above shows both limits.

## Check yourself

1. Spot is 100, `T = 1`, `σ = 20%`, both rates zero. Roughly what spot level sits at +1 standard deviation?
2. You price a call by integration and get 4.00 CCY2 pips with spot at 1.2500. What's that in CCY1%?
3. Why does the last row of the distribution table have no probability?
4. Both interest rates rise from 0% to 6%. Which way does a vanilla call's price move, and through which mechanism?
5. You integrate a *put* payoff over a distribution whose forward is far above the strike. Is the answer large or small — and what does that say about where the probability mass sits?

In [15]:
#@title Answers — run this cell to reveal
from IPython.display import Markdown
Markdown(r'''
**1.** The log return at +1sd is `μ + σ√T`. With zero rates, `μ = −σ²/2 = −0.02`, so the return is `−0.02 + 0.20 = 0.18`, giving `100·e^0.18 ≈ **118.5**`. Note it is *not* 120 — log-normal, not normal.

**2.** Divide by spot: `4.00 / 1.2500 = 3.20 CCY1%`. In basis-point language, 320 bp.

**3.** Each probability is the chance of landing in the bucket **between that row and the next**. The last row has no next row, so it bounds no bucket. If you accidentally include it as zero you get the right answer; if you shift the column by one you get a subtly wrong one, which is worse.

**4.** It **falls**. The forward depends on the *differential* `r₂ − r₁`, which is unchanged, so the distribution doesn't move. What changes is the discount factor `e^(−r₂T)` — the same expected payoff, present valued harder. (Practical C, Task A, Example 3.)

**5.** **Small.** The forward being far above the strike means the bulk of the probability sits where the put pays nothing. You're multiplying a large payoff by a tiny probability in the far left tail, and near-zero payoffs by large probabilities everywhere else. This is the same picture as the overlap chart, mirrored.
''')


**1.** The log return at +1sd is `μ + σ√T`. With zero rates, `μ = −σ²/2 = −0.02`, so the return is `−0.02 + 0.20 = 0.18`, giving `100·e^0.18 ≈ **118.5**`. Note it is *not* 120 — log-normal, not normal.

**2.** Divide by spot: `4.00 / 1.2500 = 3.20 CCY1%`. In basis-point language, 320 bp.

**3.** Each probability is the chance of landing in the bucket **between that row and the next**. The last row has no next row, so it bounds no bucket. If you accidentally include it as zero you get the right answer; if you shift the column by one you get a subtly wrong one, which is worse.

**4.** It **falls**. The forward depends on the *differential* `r₂ − r₁`, which is unchanged, so the distribution doesn't move. What changes is the discount factor `e^(−r₂T)` — the same expected payoff, present valued harder. (Practical C, Task A, Example 3.)

**5.** **Small.** The forward being far above the strike means the bulk of the probability sits where the put pays nothing. You're multiplying a large payoff by a tiny probability in the far left tail, and near-zero payoffs by large probabilities everywhere else. This is the same picture as the overlap chart, mirrored.


## Where next

**Notebook 05 — Practical C** does the same job in closed form: one expression, no grid, evaluated instantly. Having built the integration first, you'll recognise Garman–Kohlhagen as the analytic solution to exactly the integral above — and `N(d₂)` as something with a real meaning rather than an opaque symbol.

Run the cross-validation test before you move on:

```bash
pytest tests/test_cross_validation.py -v
```